# 05 — Substation names and country attribution

**Input:** `Results/4.xlsx`
**Output:** `Results/05.xlsx`, `Results/Final_Substation_Catalog.csv`

Normalises the substation fields, which mix toponyms with voltage levels,
functional descriptors and numeric identifiers. Country attribution proceeds in
order: the official country field, then codes in parentheses in the substation
name, then the project name, resolved against a manual map for the residual
cases. All affiliations are standardised to ISO-3.

In [16]:
import pandas as pd
import numpy as np
import re

In [17]:
df = pd.read_excel("Results/4.xlsx", header=[0, 1], index_col=0)

In [18]:
iso3_map = {
    'Al': 'ALB', 'Albania': 'ALB',
    'At': 'AUT', 'Austria': 'AUT',
    'Ba': 'BIH', 'Bosnia and Herzegovina': 'BIH',
    'Be': 'BEL', 'Belgium': 'BEL',
    'Bg': 'BGR', 'Bulgaria': 'BGR',
    'Ch': 'CHE', 'Switzerland': 'CHE',
    'Cy': 'CYP', 'Cyprus': 'CYP',
    'Cz': 'CZE', 'Czech republic': 'CZE', 'Czech Republic': 'CZE',
    'De': 'DEU', 'Germany': 'DEU',
    'Dk': 'DNK', 'Denmark': 'DNK',
    'Dz': 'DZA', 'Algeria': 'DZA',
    'Ee': 'EST', 'Estonia': 'EST',
    'Eg': 'EGY', 'Egypt': 'EGY',
    'Es': 'ESP', 'Spain': 'ESP',
    'Fi': 'FIN', 'Finland': 'FIN',
    'Fr': 'FRA', 'France': 'FRA',
    'Gb': 'GBR', 'Great britain': 'GBR', 'United kingdom': 'GBR', 'Uk': 'GBR',
    'Ge': 'GEO', 'Georgia': 'GEO',
    'Gr': 'GRC', 'Greece': 'GRC', 'Krete': 'GRC', 'Crete': 'GRC',
    'Hr': 'HRV', 'Croatia': 'HRV',
    'Hu': 'HUN', 'Hungary': 'HUN',
    'Ie': 'IRL', 'Ireland': 'IRL',
    'Il': 'ISR', 'Israel': 'ISR',
    'Is': 'ISL', 'Iceland': 'ISL',
    'It': 'ITA', 'Italy': 'ITA',
    'Lt': 'LTU', 'Lithuania': 'LTU',
    'Lu': 'LUX', 'Luxembourg': 'LUX',
    'Lv': 'LVA', 'Latvia': 'LVA',
    'Ly': 'LBY', 'Libya': 'LBY',
    'Ma': 'MAR', 'Morocco': 'MAR',
    'Md': 'MDA', 'Moldova': 'MDA',
    'Me': 'MNE', 'Montenegro': 'MNE',
    'Mk': 'MKD', 'Fyr of macedonia': 'MKD', 'North Macedonia': 'MKD',
    'Mt': 'MLT', 'Malta': 'MLT',
    'Nl': 'NLD', 'Netherlands': 'NLD',
    'No': 'NOR', 'Norway': 'NOR',
    'Pl': 'POL', 'Poland': 'POL',
    'Pt': 'PRT', 'Portugal': 'PRT',
    'Ro': 'ROU', 'Romania': 'ROU',
    'Rs': 'SRB', 'Serbia': 'SRB',
    'Se': 'SWE', 'Sweden': 'SWE',
    'Si': 'SVN', 'Slovenia': 'SVN',
    'Sk': 'SVK', 'Slovakia': 'SVK',
    'Tn': 'TUN', 'Tunisia': 'TUN',
    'Tr': 'TUR', 'Turkey': 'TUR',
    'Ua': 'UKR', 'Ukraine': 'UKR'
}

In [19]:
# ---------------------------------------------------------
# 1. Regex Avanzate
# ---------------------------------------------------------
parentheses_re = re.compile(r"^(.*)\((.*)\).*$")

noise_re = re.compile(r'\b(ss|station|substation|area of|hub|center|centre|offshore|power link island|\d+\s?kv|tbd|unknown)\b', re.I)

def process_substation_v3(val):
    # 1. Handle nulls, empty strings
    if pd.isna(val) or str(val).strip() == "":
        return pd.NA, pd.NA
    
    raw_s = str(val).strip()
    
    if raw_s.upper() in ["XX", "0", "-", ".", "TBD"]:
        return pd.NA, pd.NA
    
    clean_name = raw_s
    extracted_country_code = pd.NA
    
    match = parentheses_re.match(raw_s)
    if match:
        clean_name = match.group(1).strip()
        inside_content = match.group(2).strip().capitalize()
        
        extracted_country_code = iso3_map.get(inside_content, pd.NA)
        
        if pd.isna(extracted_country_code) and len(inside_content) == 2:
            extracted_country_code = inside_content.upper()
    
    if pd.isna(extracted_country_code):
        check_name = clean_name.capitalize()
        if check_name in iso3_map:
            extracted_country_code = iso3_map[check_name]

    clean_name = noise_re.sub('', clean_name)
    clean_name = re.sub(r'^\d+\s*', '', clean_name)
    clean_name = re.sub(r'[^a-zA-ZÀ-ÿ\s\.\-]', '', clean_name)
    clean_name = " ".join(clean_name.split()).title()
        
    return clean_name, extracted_country_code


years = [y for y in df.columns.levels[0] if y != 'meta']


for y in years:
    for direction in ['From', 'To']:
        orig_col = [c for c in df.columns.levels[1] 
                    if direction.lower() in c.lower() and 'substation' in c.lower() and 'clean' not in c.lower()]
        
        if orig_col and (y, orig_col[0]) in df.columns:
            source = (y, orig_col[0])
            target_name = (y, f'Inv_Substation_{direction}_Clean')
            target_country = (y, f'Extracted_Country_{direction}')
            
            results = df[source].apply(process_substation_v3)
            
            df[target_name] = results.apply(lambda x: x[0])
            df[target_country] = results.apply(lambda x: x[1])



In [20]:

for y in years:
    from_country = (y, 'Extracted_Country_From')
    to_country = (y, 'Extracted_Country_To')
    unified_col = (y, 'Extracted_Project_Country')
    
    if from_country in df.columns and to_country in df.columns:
        def merge_yearly_countries(row):
            found = set()
            val_f = row[from_country]
            val_t = row[to_country]
            
            if pd.notna(val_f): found.add(str(val_f))
            if pd.notna(val_t): found.add(str(val_t))
            
            return ";".join(sorted(list(found))) if found else pd.NA

        df[unified_col] = df.apply(merge_yearly_countries, axis=1)


In [21]:
cols_2018 = [
    ('2018', 'Project_country 1'),
    ('2018', 'Project_country 2'),
    ('2018', 'Project_country 3')
]

def merge_2018_countries(row):
    countries = []
    for col in cols_2018:
        if col in row.index:
            val = row[col]
            if pd.notna(val) and str(val).strip() != "":
                countries.append(str(val).strip())
    
    if countries:
        return "; ".join(countries)
    return np.nan

df[('2018', 'Project_Country')] = df.apply(merge_2018_countries, axis=1)

In [22]:
years_priority = ['2026', '2024', '2022', '2020', '2018', '2016', '2015', '2014', '2013', '2012', '2010']

def get_final_consolidated_country(row):
    """
    Scans through years (newest to oldest). 
    Prioritizes official 'Project_Country' over 'Extracted_Project_Country' for each year.
    Returns a semicolon-separated string of ISO3 codes.
    """
    for y in years_priority:
        official_col = (y, 'Project_Country')
        extracted_col = (y, 'Extracted_Project_Country')
        
        for col in [official_col, extracted_col]:
            if col in row.index:
                val = row[col]
                if pd.notna(val) and str(val).strip() != "" and str(val).upper() != "XX":
                    
                    clean_val = str(val).replace(',', ';').strip()
                    tokens = [t.strip().capitalize() for t in clean_val.split(';')]
                    
                    iso3_results = []
                    for t in tokens:
                        if not t: continue
                        code = iso3_map.get(t, t.upper())
                        if code not in iso3_results:
                            iso3_results.append(code)
                    
                    if iso3_results:
                        return ";".join(sorted(iso3_results))
    return pd.NA

df[('meta', 'Project_Country_ISO3')] = df.apply(get_final_consolidated_country, axis=1)

unique_final = df[('meta', 'Project_Country_ISO3')].dropna().unique()


### If the code cannot identify a country from Country column and Substation it proced searching for country in Project_Name

In [23]:
name_cols = [c for c in df.columns if c[1] == 'Project_Name']

missing_mask = df[('meta', 'Project_Country_ISO3')].isna()
df_missing = df[missing_mask].copy()

def get_most_recent_name(row):
    for col in reversed(name_cols):
        if pd.notna(row[col]) and str(row[col]).strip() != "" and str(row[col]).upper() != "XX":
            return str(row[col]).strip()
    return "Unknown Name"

missing_names_list = df_missing.apply(get_most_recent_name, axis=1).unique().tolist()

# 5. Output for review
print(f"Total projects with missing country: {len(df_missing)}")
print(f"Unique project names to analyze: {len(missing_names_list)}")
print("-" * 30)
for name in sorted(missing_names_list):
    print(name)

Total projects with missing country: 218
Unique project names to analyze: 105
------------------------------
"Northern Seas Offshore Grid infrastructure" - a Long Term Conceptual Project
2nd Offshore-Onshore Corridor (Belgium)
AC transmission line between Bordeaux and Loire Valley
AC transmission line in Grand Est region
AC transmission line in Occitania
ALEGrO
ANAI: Abengoa Northern Atlantic Interconnection
ASEI: Abengoa Southern Europe Interconnection
AT, SI, IT - South-East Alps Project
Additional project France - Spain
Aragón-Castellón
BRITIB (GB-FR-ES)
Baltic Hub
Baltic synchronization
Baltics synchro with CE
COBRA-2
CSE8 Transbalkan Corridor
Cartuja
Central Western Italy
Closing of 400 kV ring around Belgrade region
Concept project France-Switzerland 400kV AC
Concept project France-Switzerland HVDC
Connection Navarra-Basque Country
Douro Spanish-Portuguese reinforcement
Façade Atlantique
Finnish North-South reinforcement
France Germany Interconnection
Gallant
GerPol Improvements


In [24]:
black_list = ['AC', 'DC', 'HV', 'MV', 'LV', 'PC', 'OH', 'UG', 'TY', 'ND', 'XX', 'P1', 'P2']

iso3_map.update({
    'Rs': 'SRB', 'Al': 'ALB', 'Me': 'MNE', 'Ba': 'BIH', 
    'Mk': 'MKD', 'Xk': 'XKX', 'Uk': 'GBR', 'Cz': 'CZE',
    'Vierraden': 'DEU', 
    'Mikułowa': 'POL', 'Mikulowa': 'POL',
})

# Manual Map "Master"
master_manual_map = {
    "AC transmission line between Bordeaux and Loire Valley": "FRA",
    "AC transmission line in Grand Est region": "FRA",
    "AC transmission line in Occitania": "FRA",
    "HTLS in Lyonnais": "FRA",
    "Massif Central North": "FRA",
    "Façade Atlantique": "FRA",
    "Aragón-Castellón": "ESP",
    "Santa Llogaia - Bescano": "ESP",
    "Douro Spanish-Portuguese reinforcement": "ESP;PRT",
    "Limitations removal on the Italian - Slovenian border": "ITA;SVN",
    "Swiss Ellipse II": "CHE",
    "Swiss Roof": "CHE",
    "PST Hradec": "CZE",
    "PST Riddes": "CHE",
    "Closing of 400 kV ring around Belgrade region": "SRB",
    "Finnish North-South reinforcement": "FIN",
    "Southern Aegean Interconnector": "GRC",
    "SAPEI 2": "ITA",
    "Irish Scottish Links on Energy Study (ISLES)": "IRL;GBR",
    "Irish-Scottish Isles": "IRL;GBR",
    "Marex": "GBR;IRL",
    "Baltic synchronization": "EST;LVA;LTU;POL",
    "Baltic Hub": "EST;LVA;LTU",
    "COBRA-2": "DNK;NLD",
    "IberiaLink": "ESP;PRT",
    "ANAI: Abengoa Northern Atlantic Interconnection": "ESP;FRA;GBR",
    "ASEI: Abengoa Southern Europe Interconnection": "ESP;FRA;ITA",
    "Qantara Med": "FRA;TUN",
    "Greenwire Loop": "GBR;IRL",
    "Greenwire North": "GBR;IRL",
    "Greenwire South": "GBR;IRL",
    "Greenwire IE-GB": "GBR;IRL",
    "Green Energy Corridor - Phase 1": "AZE;GEO;HUN;ROU",
    "Green Energy Corridor - Phase 2": "AZE;GEO;HUN;ROU",
    "CSE8 Transbalkan Corridor": "SRB;MNE;BIH",
    "South Balkan (CSE9)": "ALB;SRB;BGR;GRC;MKD",
    "OWP Northsea TenneT Part 3": "DEU",
    "OWP Northsea TenneT Part 4": "DEU"
}


def fill_the_gaps(row):

    p_name = "Unknown"

    name_cols = [c for c in df.columns if c[1] == 'Project_Name']
    name_cols = sorted(name_cols, key=lambda x: x[0], reverse=True)
    
    for col in name_cols:
        val = row[col]
        if pd.notna(val) and str(val).strip() != "" and str(val).upper() != "XX":
            p_name = str(val).strip()
            break 
            
    if p_name in master_manual_map:
        return master_manual_map[p_name]

    found_countries = set()
    
    target_cols = [c for c in df.columns if c[1] in ['Inv_Name', 'Inv_Description', 'Project_Name']]
    target_cols = sorted(target_cols, key=lambda x: x[0], reverse=True)

    for col in target_cols:
        val = row[col]
        if pd.isna(val) or str(val).strip() == "" or str(val).upper() == "XX":
            continue
            
        text = str(val).strip()
        
        candidates_iso2 = re.findall(r'\b([a-zA-Z]{2})\b', text)
        for cand in candidates_iso2:
            if cand.upper() not in black_list:
                iso3 = iso3_map.get(cand.capitalize())
                if iso3: found_countries.add(iso3)
        
        words = re.findall(r'[a-zA-Z]{3,}', text)
        for w in words:
            iso3 = iso3_map.get(w.capitalize())
            if iso3: found_countries.add(iso3)

    if not found_countries:
        return pd.NA
    
    return ";".join(sorted(list(found_countries)))



missing_mask = df[('meta', 'Project_Country_ISO3')].isna()

if missing_mask.sum() > 0:
    df.loc[missing_mask, ('meta', 'Project_Country_ISO3')] = df[missing_mask].apply(fill_the_gaps, axis=1)

final_missing = df[('meta', 'Project_Country_ISO3')].isna().sum()


In [25]:

target_cols = [c for c in df.columns if c[1] in ['Inv_Substation_From_Clean', 'Inv_Substation_To_Clean']]

all_values = df[target_cols].values.flatten()

unique_substations = set()

for val in all_values:
    if pd.notna(val):
        s = str(val).strip()
        if s != "" and s.upper() != "XX":
            unique_substations.add(s)


unique_substations_list = sorted(list(unique_substations))

# pd.DataFrame(unique_substations_list, columns=['Substation_Name']).to_excel("Univoque_substations.xlsx", index=False)

In [26]:
years_sorted = sorted([y for y in df.columns.levels[0] if y != 'meta'], reverse=True)

for direction in ['From', 'To']:
    cols_to_check = [(y, f'Inv_Substation_{direction}_Clean') for y in years_sorted]
    
    temp_series = pd.Series(index=df.index, dtype='object')
    for col in cols_to_check:
        if col in df.columns:
            temp_series = temp_series.combine_first(df[col])
    
    df[('meta', f'Inv_Substation_{direction}_Final')] = temp_series

In [27]:
df = df.copy()

In [28]:
cols_to_remove_base = [
    'Inv_Substation_From_Clean', 
    'Extracted_Country_From', 
    'Inv_Substation_To_Clean', 
    'Extracted_Country_To', 
    'Extracted_Project_Country', 
    'Project_Country'
]

cols_to_drop = [
    col for col in df.columns 
    if col[0] != 'meta' and col[1] in cols_to_remove_base
]

df = df.drop(columns=cols_to_drop)

df = df.copy()



In [29]:
df.to_excel("Results/05.xlsx")

In [30]:
from_subs = df[[('meta', 'Inv_Substation_From_Final'), ('meta', 'Project_Country_ISO3')]].copy()
from_subs.columns = ['Substation_Name', 'Countries']

to_subs = df[[('meta', 'Inv_Substation_To_Final'), ('meta', 'Project_Country_ISO3')]].copy()
to_subs.columns = ['Substation_Name', 'Countries']

unique_substations = pd.concat([from_subs, to_subs], ignore_index=True)

unique_substations = unique_substations.dropna(subset=['Substation_Name'])
unique_substations = unique_substations.drop_duplicates().sort_values('Substation_Name')

unique_substations = unique_substations.reset_index(drop=True)

print(f"Unique Substations extracted: {len(unique_substations)}")
print(unique_substations.head(10))

unique_substations.to_csv("Results/Final_Substation_Catalog.csv", index=False, sep=';')

Unique Substations extracted: 1053
            Substation_Name Countries
0                             DEU;MAR
1                             AUT;ITA
2                             SVK;UKR
3  - Wind Park Nordergründe       DEU
4                      Aach   DEU;LUX
5                    Airolo       CHE
6                Aizkraukle   LTU;LVA
7                Albertirsa   HUN;ROU
8                    Aliano       ITA
9                  Aljarafe      <NA>
